In [1]:
import os
from pathlib import Path
import copy
import numpy as np
import torch

from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.models import build_network, load_data_to_gpu
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils
import random

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

import helpers
import importlib

importlib.reload(helpers)

/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:243: Futur

<module 'helpers' from '/storage/project/r-gchou3-0/spanse30/OpenPCDet/helpers.py'>

In [2]:
CFG_FILE = 'tools/cfgs/waymo_models/centerpoint.yaml' 
CKPT = 'output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth'  # <- put your ckpt here

cfg_from_yaml_file(CFG_FILE, cfg)
cfg.TAG = Path(CFG_FILE).stem
cfg.EXP_GROUP_PATH = 'centerpoint_waymo_demo'

logger = common_utils.create_logger()
logger.info(f'Loaded cfg from {CFG_FILE}')

dataset, test_loader, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger,
    training=False
)

len_test = len(dataset)
logger.info(f'Test set length: {len_test}')
model = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=dataset
)

logger.info(f'Loading checkpoint from: {CKPT}')
model.load_params_from_file(filename=CKPT, logger=logger, to_cpu=False)
model.cuda()
model.eval()

data_iter = iter(test_loader)
batch_dict = next(data_iter)

load_data_to_gpu(batch_dict)

# Sanity check shapes
print('points:', batch_dict['points'].shape)         # [N, 5] -> [batch_idx, x, y, z, i]
print('voxels:', batch_dict['voxels'].shape)         # [V, T, C]
print('voxel_coords:', batch_dict['voxel_coords'].shape)  # [V, 4] -> [b, z, y, x]
print('voxel_num_points:', batch_dict['voxel_num_points'].shape)  # [V]

2026-01-05 20:49:27,489   INFO  Loaded cfg from tools/cfgs/waymo_models/centerpoint.yaml
2026-01-05 20:49:27,527   INFO  Loading Waymo dataset
2026-01-05 20:49:40,168   INFO  Total skipped info 0
2026-01-05 20:49:40,169   INFO  Total samples for Waymo dataset: 38597
2026-01-05 20:49:40,170   INFO  Test set length: 38597
2026-01-05 20:49:41,680   INFO  Loading checkpoint from: output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth
2026-01-05 20:49:41,681   INFO  ==> Loading parameters from checkpoint output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth to GPU
/storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/models/detectors/detector3d_template.py:367: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https:

points: torch.Size([126501, 6])
voxels: torch.Size([75604, 5, 5])
voxel_coords: torch.Size([75604, 4])
voxel_num_points: torch.Size([75604])


In [167]:
traces = []
for i in range(0, 100):
    boxes = dataset[i]['gt_boxes']
    points = dataset[i]['points']
    mask = boxes[:, 7]==1
    boxes_car = boxes[mask]

    extracted_traces = []
    for box in boxes_car:
        cx, cy, cz = box[0], box[1], box[2]
        dx, dy, dz = box[3], box[4], box[5]
        heading = box[6]

        #put box center at origin
        shifted_points = points[:, :3] - np.array([cx, cy, cz])

        #rotate points to align with the box's local axes
        #rotate by -heading to undo boxes rotation
        cos_a = np.cos(-heading)
        sin_a = np.sin(-heading)
        R = np.array([
            [cos_a, -sin_a, 0],
            [sin_a,  cos_a, 0],
            [0,      0,     1]])
        local_points = shifted_points@R.T
        inside_mask = (
            (np.abs(local_points[:, 0]) <= dx / 2) &
            (np.abs(local_points[:, 1]) <= dy / 2) &
            (np.abs(local_points[:, 2]) <= dz / 2)
        )
        points_inside = points[inside_mask]
        if len(points_inside) > 0:
            extracted_traces.append({
                'frame': i,
                'box': box,
                'points': points_inside,
                'num_points': len(points_inside)
            })
    # print(f"=============frame {i}============")
    for trace in extracted_traces:
    # print(trace['points'])
        # print(trace['num_points'], np.linalg.norm(trace['box'][:3]))
        if(trace['num_points'] <= 100): traces.append(trace)

# for i, trace in enumerate(traces):
#     print(i, trace['frame'], trace['num_points'])

0 0 12
1 1 93
2 2 34
3 3 31
4 3 32
5 4 53
6 4 42
7 5 39
8 5 44
9 6 36
10 6 49
11 7 32
12 7 16
13 7 60
14 8 11
15 8 71
16 9 5
17 9 65
18 9 72
19 10 2
20 10 82
21 10 91
22 11 2
23 11 68
24 12 58
25 13 97
26 13 15
27 13 47
28 14 95
29 14 29
30 14 44
31 15 85
32 15 71
33 15 7
34 15 46
35 15 37
36 16 55
37 16 46
38 16 19
39 16 55
40 16 33
41 17 46
42 17 18
43 17 21
44 17 83
45 18 36
46 18 23
47 19 39
48 19 25
49 20 24
50 20 32
51 21 18
52 21 38
53 22 18
54 22 58
55 23 14
56 23 69
57 24 86
58 26 80
59 26 90
60 27 73
61 28 80
62 28 50
63 29 12
64 29 59
65 30 75
66 30 42
67 31 39
68 31 58
69 32 24
70 32 96
71 33 19
72 34 28
73 34 94
74 35 13
75 35 77
76 36 32
77 36 9
78 36 66
79 37 43
80 37 28
81 37 57
82 38 47
83 38 33
84 38 45
85 39 49
86 39 71
87 39 26
88 39 33
89 40 33
90 40 80
91 40 81
92 40 23
93 40 42
94 41 43
95 41 59
96 41 27
97 41 100
98 41 7
99 42 31
100 42 55
101 42 14
102 43 52
103 43 33
104 43 44
105 44 72
106 44 46
107 44 40
108 45 82
109 45 41
110 46 91
111 46 32
112 47 76
113 

In [144]:
enumerate(traces)

In [147]:
# TODO: ONLY ROTATE IF ORIGINAL TRACE POSITION IS BEHIND EGO
selected_trace = traces[11]
centered_points_trace = selected_trace['points'][:, :2] - selected_trace['box'][:2]
position = np.array([15, -3])

angle = 0
if(selected_trace['box'][0]<0):
    print("rear object")
    angle = np.pi

c = np.cos(angle)
s = np.sin(angle)
R = np.array([[c, -s],
              [s, c]])


rotated_trace = centered_points_trace @ R.T
front_near_trace_planar = rotated_trace + position
front_near_trace = np.concatenate([front_near_trace_planar, selected_trace['points'][:, 2].reshape(-1, 1)], axis = 1)

# helpers.plot_trace(front_near_trace)

rear object


In [148]:
importlib.reload(helpers)
frame = 0
r_trace, theta_trace, phi_trace = helpers.cart_2_spherical(front_near_trace)
scene_points = dataset[frame]['points'][:, :3]
r_scene, theta_scene, phi_scene = helpers.cart_2_spherical(scene_points)

trace_spherical = np.concatenate([r_trace.reshape(-1, 1), theta_trace.reshape(-1, 1), phi_trace.reshape(-1, 1)], axis = 1)
scene_spherical = np.concatenate([r_scene.reshape(-1, 1), theta_scene.reshape(-1, 1), phi_scene.reshape(-1, 1)], axis = 1)

azimuths = np.deg2rad(np.arange(0, 360.1, np.deg2rad(0.1358)))
elevations = np.load('waymo_top_lidar_inclinations.npy')

from collections import defaultdict

Grid = defaultdict(list)
dtheta = np.deg2rad(0.1358)
for idx, (r, theta, phi) in enumerate(scene_spherical):
    az = theta % (2*np.pi)
    az_idx = int(np.floor(az/dtheta))
    beam_idx = np.argmin(np.abs(elevations-phi))
    Grid[(az_idx, beam_idx)].append((r, idx))
for key in Grid:
    Grid[key].sort(key=lambda x: x[0])  # sort by range
    

remove_spoof = []
remove_scene = []
for idx, (r, theta, phi) in enumerate(trace_spherical):
    az = theta % (2*np.pi)
    az_idx = int(np.floor(az/dtheta))
    beam_idx = np.argmin(np.abs(elevations-phi))
    returns = Grid[(az_idx, beam_idx)]
    if(len(returns) == 0):
        continue
    else:
        # spoof point is behind first real return, remove it
        if returns[0][0] < r:
            print(round(azimuths[az_idx], 4), round(elevations[beam_idx], 4), round(r, 4), returns[0][0], returns[0][1])
            #ranges are already sorted in ascending order
            remove_spoof.append(idx)
        #spoof is front of all real returns
        elif r < returns[0][0]:
            for element in returns:
                remove_scene.append(element[1]) #remove all scene points 
        #spoof is between two real returns
        else:
            for k, (r_scene, idx_scene) in enumerate(returns):
                if r_scene > r:
                    # keep spoof
                    for element in returns[k:]:
                        remove_scene.append(element[1])
                    break 
print(remove_spoof)
            
    

fn_points = np.concatenate([front_near_trace, selected_trace['points'][:, -2:]], axis = 1)
scene_points = dataset[frame]['points'] 

mask_remove_spoof = np.ones(fn_points.shape[0], dtype=bool)
mask_remove_spoof[remove_spoof] = False
fn_points_rc = fn_points[mask_remove_spoof]

mask_remove_scene = np.ones(scene_points.shape[0], dtype=bool)
mask_remove_scene[remove_scene] = False
scene_points_rc = scene_points[mask_remove_scene]

frame_points_spoof_rc = np.concatenate([scene_points_rc, fn_points_rc], axis = 0)

[]


In [149]:
# for remove in remove_spoof:
#     print(front_near_trace[remove][-1])
#     # print(trace_spherical[remove])

In [150]:
# helpers.plot_trace(fn_points[:, :3])

# helpers.plot_trace(fn_points_rc[:, :3])

In [151]:
# dict_mod = dataset[frame]
# dict_mod['points'] = frame_points_spoof_rc
# print(dict_mod['points'].shape)

# importlib.reload(helpers)
# helpers.plot_frame(dict_mod, dataset, frame, 0, 1, 150000)


In [152]:
import numpy as np


# 1. Define mapping helper
# Waymo labels: 1=Vehicle, 2=Pedestrian, 3=Cyclist
# dataset.class_names is typically ['Vehicle', 'Pedestrian', 'Cyclist']
def inject_gt_names(data_dict, class_names):
    if 'gt_boxes' in data_dict and 'gt_names' not in data_dict:
        # Extract the last column (label index)
        # gt_boxes shape is (N, 8), index 7 is the label
        labels = data_dict['gt_boxes'][:, -1].astype(int)
        
        # Map index to name (Label 1 -> Index 0)
        # We use l-1 because Waymo labels are 1-based
        names = np.array([class_names[l - 1] for l in labels])
        data_dict['gt_names'] = names
    return data_dict

# 2. Reload fresh data to be safe (aliasing prevention)
dict_clean = dataset[frame]
dict_mod = dataset[frame] # Currently identical to clean

# 3. Inject the spoofed points
# Ensure shape is (N, 4) or (N, 5) matching the original
dict_mod['points'] = frame_points_spoof_rc

# 4. Remove old voxel data 
# (Since we changed points, old voxels are invalid. 
# prepare_data usually overwrites, but this prevents shape mismatch errors)
for k in ['voxels', 'voxel_coords', 'voxel_num_points']:
    dict_mod.pop(k, None)

# 5. Inject the missing 'gt_names'
dict_clean = inject_gt_names(dict_clean, dataset.class_names)
dict_mod   = inject_gt_names(dict_mod, dataset.class_names)

# 6. Now you can safely re-voxelize
dict_clean = dataset.prepare_data(dict_clean)
dict_mod   = dataset.prepare_data(dict_mod)

dict_clean['batch_size'] = 1
dict_mod['batch_size']   = 1

# 3. Load the BATCH to GPU
# (Do this on the batch, not the single dicts)
load_data_to_gpu(dict_clean)
load_data_to_gpu(dict_mod)
def convert_to_batch(dict_):
    data_dict = copy.deepcopy(dict_)
    device = dict_['points'].device
    data_dict['sample_idx'] = torch.tensor([dict_['sample_idx']], device = device)
    rows_points = dict_['points'].shape[0]
    zeros_points =torch.zeros((rows_points, 1), device = device)
    data_dict['points'] = torch.cat((zeros_points, dict_['points']), dim = 1)
    data_dict['frame_id'] = [dict_['frame_id']]
    data_dict['gt_boxes'] = dict_['gt_boxes'].unsqueeze(0)
    data_dict['lidar_aug_matrix'] = dict_['lidar_aug_matrix'].unsqueeze(0)
    data_dict['use_lead_xyz'] = torch.tensor([1], device = device) if dict_['use_lead_xyz'] else torch.tensor([0], device = device)
    rows_vc = dict_['voxel_coords'].shape[0]
    zeros_vc = torch.zeros((rows_vc, 1), device = device)
    data_dict['voxel_coords'] = torch.cat((zeros_vc, dict_['voxel_coords']), dim = 1)
    data_dict['metadata'] = [dict_['metadata']]
    
    return data_dict

    

batch_clean = convert_to_batch(dict_clean)
batch_mod = convert_to_batch(dict_mod)

model.eval()
with torch.no_grad():
    pred_clean, _ = model.forward(batch_clean)
    pred_mod, _   = model.forward(batch_mod)
print("clean num boxes:", pred_clean[0]['pred_boxes'].shape[0])
print("mod   num boxes:", pred_mod[0]['pred_boxes'].shape[0])
# Look at the raw scores before filtering
raw_scores = pred_mod[0]['pred_scores']
print(f"Max confidence score in frame: {raw_scores.max().item():.4f}")

candidate = []
for i in range (0, pred_mod[0]['pred_boxes'].shape[0]):
    x = pred_mod[0]['pred_boxes'][:, 0]
    y = pred_mod[0]['pred_boxes'][:, 1]
    if(x[i] >= 13 and x[i]<=17 and y[i]>=-5 and y[i]<=-1):
        candidate.append(i)
        print("prediction: ", i," |box location: ",  pred_mod[0]['pred_boxes'][i], " |score: ", pred_mod[0]['pred_scores'][3], " |label: ", pred_mod[0]['pred_labels'][3])


clean num boxes: 97
mod   num boxes: 99
Max confidence score in frame: 0.9254
prediction:  7  |box location:  tensor([14.7144, -3.0372,  1.2687,  4.4955,  2.0924,  1.7817, -0.1029],
       device='cuda:0')  |score:  tensor(0.8851, device='cuda:0')  |label:  tensor(1, device='cuda:0')
prediction:  57  |box location:  tensor([14.3456, -3.2423,  1.2856,  3.9894,  1.9025,  1.6600, -0.2888],
       device='cuda:0')  |score:  tensor(0.8851, device='cuda:0')  |label:  tensor(1, device='cuda:0')


In [166]:
#new logic: compare distance of centroid to centroids of predicted boxes to see which one is closest
# also compare box volume (trace box volume vs pred boxes volume) to see which one matches the closest. 
# if these two match then there is confirmed box at location of trace
# check its classification and score. 


pos_t = torch.from_numpy(position).to('cuda')
val_t = torch.tensor([selected_trace['box'][2]], device='cuda')
box_dim = torch.from_numpy(selected_trace['box'][3:6]).to('cuda')
trace_box_c = torch.cat((pos_t, val_t, box_dim))
d_trace = torch.linalg.norm(trace_box_c[:3])
V_trace = selected_trace['box'][3]*selected_trace['box'][4]*selected_trace['box'][5]

comparisons = []
def cosine_similarity(a, b, eps=1e-8):
    return torch.dot(a, b) / (torch.linalg.norm(a) * torch.linalg.norm(b) + eps)
for i in range (0, pred_mod[0]['pred_boxes'].shape[0]):
    box = pred_mod[0]['pred_boxes'][i][:3]
    dim = pred_mod[0]['pred_boxes'][i][3:6]
    d_pred = torch.linalg.norm(box)
    V_pred = dim[0]*dim[1]*dim[2]
    label_pred = pred_mod[0]['pred_labels'][i].item()
    score_pred = pred_mod[0]['pred_scores'][i].item()
    
    d_d = np.round(torch.abs(d_pred - d_trace).item(), 2)
    d_V = np.round(torch.abs(V_pred - V_trace).item(), 2)
    d_label = label_pred - 1
   
    cos = cosine_similarity(trace_box_c[:3].float(), box)
    data = {
        "distance" : d_d,
        "cos_sim" : cos,
        "volume" : d_V,
        "label" : d_label,
        "score" : score_pred
    }
    comparisons.append(data)
    # print(f"index = {i} | d_distance: {d_d} | cos similarity: {np.round(cos.item(), 2)} | d_Volume:{d_V} | d_label: {d_label} | score: {np.round(score_pred, 2)}")

In [165]:
best_idx = sorted(
    range(len(comparisons)),
    key=lambda i: (
        comparisons[i]["distance"],
        -comparisons[i]["cos_sim"],
        comparisons[i]["volume"]
    )
)[0]

print("Best match index:", best_idx)
print(comparisons[best_idx])
print(pred_mod[0]['pred_boxes'][best_idx])
print(trace_box_c)

Best match index: 7
{'distance': 0.27, 'cos_sim': tensor(1.0000, device='cuda:0'), 'volume': 2.95, 'label': 0, 'score': 0.6840183138847351}
tensor([14.7144, -3.0372,  1.2687,  4.4955,  2.0924,  1.7817, -0.1029],
       device='cuda:0')
tensor([15.0000, -3.0000,  1.2102,  5.0048,  1.9428,  1.4200], device='cuda:0',
       dtype=torch.float64)


array([5.00478114, 1.94275874, 1.42      ])

In [79]:
# npoints = front_near_trace.shape[0]
# xc = sum(front_near_trace[:, 0])/npoints
# yc = sum(front_near_trace[:, 1])/npoints
# zc = sum(front_near_trace[:, 2])/npoints

# trace_avg_c = torch.tensor([xc, yc, zc], device = 'cuda')

# trace_avg_c

In [12]:
# from waymo_open_dataset import dataset_pb2
# import tensorflow as tf

# # Example: Read from a TFRecord file
# FILENAME = 'individual_files_validation_segment-10203656353524179475_7625_000_7645_000_with_camera_labels.tfrecord'
# dataset = tf.data.TFRecordDataset(FILENAME, compression_type='')

# for data in dataset:
#     frame = dataset_pb2.Frame()
#     frame.ParseFromString(bytearray(data.numpy()))

#     # 1. Access the Context (metadata for this segment)
#     context = frame.context

#     # 2. Iterate through calibrations to find the Top LiDAR
#     for calibration in context.laser_calibrations:
#         if calibration.name == dataset_pb2.LaserName.TOP:
            
#             # 3. Access the beam_inclinations
#             # This is a list of elevation angles (in radians)
#             # The length matches the height of the range image (e.g., 64)
#             beam_inclinations = calibration.beam_inclinations
            
#             print(f"Found {len(beam_inclinations)} beams for Top LiDAR.")
#             # print(f"First 5 angles (radians): {beam_inclinations[:5]}")
#             print(beam_inclinations)
            
#             # Note: If this list is empty, you must fallback to calculating 
#             # linear steps using calibration.beam_inclination_min/max, 
#             # but for the Top LiDAR, this list should always be populated.
#             break
#     break

Found 64 beams for Top LiDAR.
[-0.3095214987479722, -0.2985423857695544, -0.28781847765244195, -0.27766609201893244, -0.26699100555316435, -0.2573929091807843, -0.24763909092297332, -0.2380383506951802, -0.22886440353835735, -0.21964150400487803, -0.21015745652833395, -0.20137599845004805, -0.19293571769984585, -0.18455331616243953, -0.17620463519698948, -0.16808360354921748, -0.1603510019650427, -0.15250658370550685, -0.14506482502630758, -0.1375849409165577, -0.13040110193186694, -0.1231675702476438, -0.1164231942236782, -0.10988995881236963, -0.103787914864375, -0.09745489855894918, -0.09157780502236523, -0.08558255013343885, -0.0801864882429475, -0.07478464850680977, -0.0697650415235831, -0.06472732715010787, -0.059973591683473826, -0.05512564171884393, -0.0507967308565207, -0.04659180754260439, -0.042516325036812797, -0.03844255122699325, -0.0345103165665126, -0.03111472124628767, -0.027910325083470244, -0.024802625308065096, -0.021897645451992354, -0.018816731162618616, -0.015806

In [13]:
beam_inclinations

[-0.3095214987479722, -0.2985423857695544, -0.28781847765244195, -0.27766609201893244, -0.26699100555316435, -0.2573929091807843, -0.24763909092297332, -0.2380383506951802, -0.22886440353835735, -0.21964150400487803, -0.21015745652833395, -0.20137599845004805, -0.19293571769984585, -0.18455331616243953, -0.17620463519698948, -0.16808360354921748, -0.1603510019650427, -0.15250658370550685, -0.14506482502630758, -0.1375849409165577, -0.13040110193186694, -0.1231675702476438, -0.1164231942236782, -0.10988995881236963, -0.103787914864375, -0.09745489855894918, -0.09157780502236523, -0.08558255013343885, -0.0801864882429475, -0.07478464850680977, -0.0697650415235831, -0.06472732715010787, -0.059973591683473826, -0.05512564171884393, -0.0507967308565207, -0.04659180754260439, -0.042516325036812797, -0.03844255122699325, -0.0345103165665126, -0.03111472124628767, -0.027910325083470244, -0.024802625308065096, -0.021897645451992354, -0.018816731162618616, -0.015806880446189275, -0.0128777471528

In [15]:
data_to_save = np.array(beam_inclinations)

# Save to file
np.save('waymo_top_lidar_inclinations.npy', data_to_save)